# Phase 1 six-question report v2: means and return-depth thresholds

Reads the saved `phase1_chronological_paths.csv`. No raw ticker download or trade replay is needed. The primary event is already the first confirmed breakout per ticker/date/timeframe. The 30-minute OR and 1/2/5-minute confirmation definitions are unchanged.

Return = one-minute wick **touches** the broken OR line after signal close. Close inside = a subsequent one-minute close inside the OR. Extension before touch excludes the touch minute because OHLC ordering is unknown. Penetration after touch is missing when there was no touch.

All percentages are relative to the broken OR boundary. Post-return extension and penetration may both occur; this file contains their maxima, not their first-hit order. We report 'both' explicitly, and do not infer which move happened first. For a truly sequential next-trade rule, replay the original minute bars.
In report 01, threshold rates have both denominators: among OR returners and among all confirmed breakouts.


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/OR_Phase1_Chronology_Stop_Target')
SRC=BASE/'phase1_chronological_paths.csv'
assert SRC.exists(),f'Missing {SRC}; run Phase 1 v2 first'
d=pd.read_csv(SRC,dtype={'ticker':str,'date':str})
OUT=BASE/'Six_Questions'
OUT.mkdir(exist_ok=True)
for c in ['return_touch','return_close_inside']:
    d[c]=d[c].astype(str).str.lower().eq('true')
d['timeframe_min']=pd.to_numeric(d.timeframe_min)
d['horizon']=d.horizon.astype(str)
d['event_id']=d.ticker+'|'+d.date+'|'+d.timeframe_min.astype(str)
assert not d.duplicated(['event_id','horizon']).any()
print('Saved window rows',len(d),'first-event IDs',d.event_id.nunique(),
      'tickers',d.ticker.nunique(),'dates',d.date.nunique())
print('Horizons',sorted(d.horizon.unique()))
display(d.head(3))

In [ ]:
# 1. Before-return extension and 3. after-return penetration.
def q13(g):
    returned=g[g.return_touch]
    def quant(s,q):return s.quantile(q)
    return pd.Series({
      'events':len(g),'tickers':g.ticker.nunique(),'dates':g.date.nunique(),
      'returners':len(returned),'return_pct':100*g.return_touch.mean(),
      'pre_return_extension_mean_pct':g.extension_before_return_pct.mean(),
      'pre_return_extension_median_pct':g.extension_before_return_pct.median(),
      'pre_return_extension_p75_pct':quant(g.extension_before_return_pct,.75),
      'pre_return_extension_p90_pct':quant(g.extension_before_return_pct,.90),
      'zero_pre_return_extension_pct':100*g.extension_before_return_pct.eq(0).mean(),
      'pre_return_extension_median_OR_widths':g.extension_before_return_or_width.median(),
      'penetration_mean_pct_returners':returned.penetration_after_return_pct.mean(),
      'penetration_median_pct_returners':returned.penetration_after_return_pct.median(),
      'penetration_p75_pct_returners':quant(returned.penetration_after_return_pct,.75),
      'penetration_p90_pct_returners':quant(returned.penetration_after_return_pct,.90),
      'penetration_median_OR_widths_returners':returned.penetration_after_return_or_width.median(),
      'penetration_at_least_0_25pct_returners':100*returned.penetration_after_return_pct.ge(.25).mean(),
      'penetration_at_least_0_5pct_returners':100*returned.penetration_after_return_pct.ge(.5).mean(),
      'penetration_at_least_0_75pct_returners':100*returned.penetration_after_return_pct.ge(.75).mean(),
      'penetration_at_least_1pct_returners':100*returned.penetration_after_return_pct.ge(1).mean(),
      'penetration_at_least_0_5pct_all_breakouts':100*g.penetration_after_return_pct.ge(.5).mean(),
      'penetration_at_least_0_75pct_all_breakouts':100*g.penetration_after_return_pct.ge(.75).mean(),
      'penetration_at_least_1pct_all_breakouts':100*g.penetration_after_return_pct.ge(1).mean(),
    })
q13t=d.groupby(['timeframe_min','side','horizon'],dropna=False).apply(q13,include_groups=False).reset_index()
q13t.to_csv(OUT/'01_extension_03_penetration.csv',index=False)
display(q13t.round(2))

In [ ]:
# 2. Return within 5, 15, 30, 60 minutes.
# Use ONE 60-minute-complete cohort for all deadlines, avoiding changing denominators.
sixty=d[d.horizon.eq('60')].copy()
def q2(g):
    return pd.Series({'events_with_complete_60m':len(g),'tickers':g.ticker.nunique(),
      'dates':g.date.nunique(),**{
      f'returned_within_{m}m':int((g.return_touch&g.minutes_to_touch.le(m)).sum())
          for m in (5,15,30,60)},
      **{f'returned_within_{m}m_pct':round(100*(g.return_touch&g.minutes_to_touch.le(m)).mean(),1)
          for m in (5,15,30,60)}})
q2t=sixty.groupby(['timeframe_min','side'],dropna=False).apply(q2,include_groups=False).reset_index()
q2t.to_csv(OUT/'02_return_by_deadline_same_cohort.csv',index=False)
display(q2t)
print('An event on minute 5 is counted in the next interval; deadline means at or before minute 5/15/30/60.')

In [ ]:
# 4. Wick touch vs close back inside (same exact event cohorts).
def q4(g):
    touched=g[g.return_touch]
    return pd.Series({'events':len(g),'wick_touches':len(touched),
      'wick_touch_pct_all':100*g.return_touch.mean(),
      'closes_inside':int(g.return_close_inside.sum()),
      'close_inside_pct_all':100*g.return_close_inside.mean(),
      'touch_only_no_close':int((g.return_touch&~g.return_close_inside).sum()),
      'touch_only_pct_of_touches':100*(~touched.return_close_inside).mean() if len(touched) else np.nan,
      'closes_inside_without_recorded_touch':int((~g.return_touch&g.return_close_inside).sum())})
q4t=d.groupby(['timeframe_min','side','horizon'],dropna=False).apply(q4,include_groups=False).reset_index()
q4t.to_csv(OUT/'04_touch_vs_close_inside.csv',index=False)
display(q4t.round(2))

In [ ]:
# 5. After first return: price can extend outward AND penetrate inward.
# A +0.5%-of-boundary outward move is a declared descriptive threshold, not a trading rule.
z=d[d.return_touch].copy()
z['post_return_extension_pct']=100*z.extension_after_return_dollars/z.boundary
z['resume_0_5pct']=z.post_return_extension_pct.ge(.5)
z['opposite_OR']=z.penetration_after_return_or_width.ge(1)
z['after_return_bucket']=np.select(
 [z.resume_0_5pct & z.opposite_OR,
  z.resume_0_5pct & ~z.opposite_OR,
  ~z.resume_0_5pct & z.opposite_OR],
 ['both','resume_only','opposite_only'],default='neither')
def q5(g):
    return pd.Series({'returned_events':len(g),'tickers':g.ticker.nunique(),
      'dates':g.date.nunique(),
      'post_return_extension_median_pct':g.post_return_extension_pct.median(),
      'post_return_extension_p75_pct':g.post_return_extension_pct.quantile(.75),
      'resume_0_5pct':int(g.resume_0_5pct.sum()),
      'resume_0_5pct_rate':100*g.resume_0_5pct.mean(),
      'opposite_OR':int(g.opposite_OR.sum()),
      'opposite_OR_rate':100*g.opposite_OR.mean(),
      **{label:int(g.after_return_bucket.eq(label).sum())
        for label in ['resume_only','opposite_only','both','neither']}})
q5t=z.groupby(['timeframe_min','side','horizon'],dropna=False).apply(q5,include_groups=False).reset_index()
q5t.to_csv(OUT/'05_after_return_paths.csv',index=False)
display(q5t.round(2))
print('Both means both occurred by the horizon; the saved maxima cannot tell which came first.')

In [ ]:
# 6. Common eligible ticker-days AND matched ticker-day-side, evaluated at the 60-minute horizon.
p=sixty.copy()
p['daykey']=p.ticker+'|'+p.date
common=set.intersection(*(set(p.loc[p.timeframe_min.eq(tf),'daykey']) for tf in (1,2,5)))
eligible=p[p.daykey.isin(common)].copy()
print('Common eligible ticker-days:',len(common),
      '| first-event counts by timeframe:',eligible.timeframe_min.value_counts().sort_index().to_dict())
paired=eligible.groupby(['timeframe_min','side'],dropna=False).apply(q13,include_groups=False).reset_index()
paired.to_csv(OUT/'06_common_eligible_days_by_side.csv',index=False)
display(paired.round(2))
# Stricter paired comparison: same ticker/date and same first-break direction in all 3.
pivot=eligible.pivot(index='daykey',columns='timeframe_min',values='side')
same=pivot.dropna().loc[lambda x:(x[1]==x[2])&(x[2]==x[5])].index
strict=eligible[eligible.daykey.isin(same)].copy()
strict_summary=strict.groupby(['timeframe_min','side'],dropna=False).apply(q13,include_groups=False).reset_index()
strict_summary.to_csv(OUT/'06_matched_first_direction.csv',index=False)
print('Ticker-days with the same first breakout direction in all 3:',len(same))
display(strict_summary.round(2))
# Number and reasons for disagreements, without treating three frames as independent trades.
directions=eligible.pivot(index='daykey',columns='timeframe_min',values='side')
print('Different first-break direction across frames:',
      int((directions.nunique(axis=1)>1).sum()))
print('Saved six reports to:',OUT)

## How to read these reports

- Report 01 has extensions before the first touch and depth after it.
- Report 02 uses the identical complete-60-minute cohort for all four deadlines, including five minutes.
- Report 04 shows how many wick touches did **not** become a close back inside.
- Report 05 separates resume-only, opposite-only, both, and neither; it does not imply which happened first.
- Report 06 compares common ticker-days, then the stricter set with the same first breakout direction across 1/2/5-minute confirmations.

The 15/30/60/session-end reports have different denominators because the saved Phase 1 analysis requires complete one-minute bars for each window. Do not subtract counts from different rows to infer a new return rate. A no-return result means no return within that measured horizon.